# Mathematical Derivations in Code

This notebook links the mathematical foundations of the pricing engine to executable numerical experiments. It uses the reusable `derivatives_engine` package rather than re-implementing pricing logic in notebook cells.

The experiments cover risk-neutral GBM simulation, discounted payoff Monte Carlo pricing, numerical convergence, finite-difference Greek validation and a compact Heston path demonstration.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

FIGURE_DIR = ROOT / "docs" / "images"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

from derivatives_engine.models.black_scholes import price as black_scholes_price
from derivatives_engine.models.heston import HestonParams, simulate_heston_paths
from derivatives_engine.models.monte_carlo import price_european_option, simulate_gbm_paths
from derivatives_engine.utils.convergence import (
    binomial_convergence_results,
    greek_comparison_results,
    monte_carlo_convergence_results,
)

pd.set_option("display.precision", 6)
plt.style.use("seaborn-v0_8-whitegrid")

## Model Setup

The base case uses the same benchmark parameters as the validation report: an at-the-money one-year European option with 20% volatility and a 5% risk-free rate. Under the risk-neutral measure, the drift is `r - q`; the expected discounted payoff is the option value.

In [ ]:
S0 = 100.0
K = 100.0
T = 1.0
r = 0.05
q = 0.0
sigma = 0.20
option_type = "call"

bs_call = black_scholes_price(S0, K, T, r, q, sigma, option_type)
print(f"Black-Scholes benchmark call price: {bs_call:.6f}")

## 1. GBM Path Simulation

The Monte Carlo module simulates the exact lognormal discretisation of risk-neutral geometric Brownian motion:

```text
dS_t = (r - q) S_t dt + sigma S_t dW_t^Q
```

The plot below shows sample paths. The randomness is driven by Brownian shocks; the deterministic drift term is the risk-neutral carry after dividend yield.

In [ ]:
gbm_paths = simulate_gbm_paths(
    S0, T, r, q, sigma, n_paths=2_000, n_steps=252, seed=42
)
time_grid = [i * T / 252 for i in range(253)]

fig, ax = plt.subplots(figsize=(9, 5))
for path in gbm_paths[:25]:
    ax.plot(time_grid, path, linewidth=0.8, alpha=0.65)
ax.axhline(K, color="black", linestyle="--", linewidth=1.0, label="Strike")
ax.set_title("Risk-Neutral GBM Sample Paths")
ax.set_xlabel("Time to maturity (years)")
ax.set_ylabel("Underlying price")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "mathematical_derivations_gbm_paths.png", dpi=160)
plt.show()

## 2. Discounted Payoff Monte Carlo Pricing

A European call can be priced as the discounted sample average of terminal payoffs:

```text
V_0 ? exp(-rT) * (1 / N) * sum(max(S_T - K, 0))
```

The estimator is unbiased for the simulated model, but it has sampling error. The implementation reports a standard error and confidence interval alongside the price.

In [ ]:
mc_result = price_european_option(
    S0, K, T, r, q, sigma, option_type, n_paths=100_000, seed=123
)
summary = pd.DataFrame(
    [
        {
            "method": mc_result.method,
            "mc_price": mc_result.price,
            "black_scholes_price": bs_call,
            "absolute_error": abs(mc_result.price - bs_call),
            "standard_error": mc_result.standard_error,
            "ci_lower": mc_result.confidence_interval[0],
            "ci_upper": mc_result.confidence_interval[1],
        }
    ]
)
summary

## 3. Monte Carlo Convergence

Monte Carlo convergence is statistical rather than monotonic. The standard error scales as `O(1 / sqrt(N))`, so reducing error by a factor of 10 generally requires about 100 times as many paths.

In [ ]:
mc_convergence = monte_carlo_convergence_results(
    path_counts=(1_000, 5_000, 10_000, 50_000, 100_000), seed=2026
)
mc_convergence

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(
    mc_convergence["paths"],
    mc_convergence["absolute_error"],
    marker="o",
    label="absolute pricing error",
)
ax.loglog(
    mc_convergence["paths"],
    mc_convergence["standard_error"],
    marker="s",
    label="standard error",
)
ax.set_title("Monte Carlo Error vs Number of Paths")
ax.set_xlabel("Number of paths")
ax.set_ylabel("Error / standard error")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "mathematical_derivations_mc_convergence.png", dpi=160)
plt.show()

## 4. CRR Binomial Tree Convergence

The Cox-Ross-Rubinstein tree discretises the same risk-neutral GBM dynamics with up/down factors and backward induction. For European vanilla options, the tree price should converge toward the closed-form Black-Scholes value as the number of time steps increases.

In [ ]:
binomial_convergence = binomial_convergence_results(
    steps=(25, 50, 100, 250, 500, 1_000)
)
binomial_convergence

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(
    binomial_convergence["steps"],
    binomial_convergence["absolute_error"],
    marker="o",
)
ax.set_title("CRR Binomial Convergence to Black-Scholes")
ax.set_xlabel("Tree steps")
ax.set_ylabel("Absolute error")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "mathematical_derivations_binomial_convergence.png", dpi=160)
plt.show()

## 5. Analytical vs Finite-Difference Greeks

Analytical Greeks come from differentiating the pricing formula. Finite-difference Greeks bump inputs and revalue the option, which makes them useful as independent numerical checks. Small differences are expected due to bump size and floating-point precision.

In [ ]:
greeks = greek_comparison_results()
greeks[[
    "greek",
    "analytical",
    "finite_difference",
    "absolute_error",
    "relative_error",
]]

## 6. Heston Path Simulation

The Heston model introduces stochastic variance. Negative spot/variance correlation can generate equity-index-style skew, while mean reversion pulls variance toward a long-run level. This notebook only visualises paths; calibration evidence remains in the package examples and validation reports.

In [ ]:
heston_params = HestonParams(v0=0.04, kappa=1.5, theta=0.04, sigma_v=0.35, rho=-0.6)
heston_paths = simulate_heston_paths(
    S0, T, r, q, heston_params, n_paths=800, n_steps=252, seed=314
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharex=True)
for path in heston_paths.spot_paths[:20]:
    axes[0].plot(heston_paths.time_grid, path, linewidth=0.8, alpha=0.65)
axes[0].set_title("Heston Spot Paths")
axes[0].set_xlabel("Time to maturity (years)")
axes[0].set_ylabel("Underlying price")

for var_path in heston_paths.variance_paths[:20]:
    axes[1].plot(heston_paths.time_grid, var_path, linewidth=0.8, alpha=0.65)
axes[1].set_title("Heston Variance Paths")
axes[1].set_xlabel("Time to maturity (years)")
axes[1].set_ylabel("Variance")

fig.tight_layout()
fig.savefig(FIGURE_DIR / "mathematical_derivations_heston_paths.png", dpi=160)
plt.show()

## Summary

The experiments mirror the repository's validation philosophy: derive or reference the analytical benchmark where possible, use numerical methods for contracts and dynamics beyond closed form, report uncertainty for stochastic estimators and treat synthetic demonstrations as validation-aware evidence rather than formal model approval.